# Advanced packages - UZF (Unsaturated Zone Flow)

Part of the **synthetic-valley advanced-packages** series, which upgrades a calibrated base model's simple boundary conditions to MODFLOW 6 *advanced packages* one notebook at a time. Each notebook loads the shared advanced model from `models/` (creating it from the base model on first use with `notebook_helpers.load_or_create_advanced_model`) and adds one package, so they can be run independently and in any order - except where a package depends on others.

- **UZF** (`advanced-packages-uzf`) - recharge (RCH) -> Unsaturated Zone Flow *(this notebook)*
- **MAW** (`advanced-packages-maw`) - wells (WEL) -> Multi-Aquifer Well
- **SFR** (`advanced-packages-sfr`) - river (RIV) -> Streamflow Routing
- **LAK** (`advanced-packages-lak`) - high-K lake -> Lake package
- **MVR** (`advanced-packages-mvr`) - Water Mover (requires UZF, LAK, SFR)
- **Processing output** (`advanced-packages-processing`) - run the model and plot results

Replace the recharge (RCH) package with the Unsaturated Zone Flow (UZF) package, which routes precipitation and evapotranspiration through an unsaturated column above the water table. Rejected or discharged UZF water can later be passed to other packages by the Mover (see the MVR notebook).

## Imports and setup

Import FloPy and the helpers, then choose the temporal resolution
(`sample_frequency`) and the model name.

In [ ]:
import flopy
from notebook_helpers import (
    IN2FT,
    drop_packages,
    load_or_create_advanced_model,
    load_spatial_data,
    load_temporal_data,
)

In [ ]:
sample_frequency = "annual"  # monthly or annual
name = "sv"

## Load or create the advanced model

Load the shared advanced model from `models/` (or create it from the base model on first use), then read the grid dimensions and the forcing and lake data this package needs.

In [ ]:
sim = load_or_create_advanced_model(sample_frequency, name)
gwf = sim.get_model(name)
nper = sim.tdis.nper.array
nrow, ncol = gwf.dis.nrow.array, gwf.dis.ncol.array

In [ ]:
temporal_df = load_temporal_data(sample_frequency)
_, lake_location, _ = load_spatial_data()

## Build the UZF package

One UZF cell per active (non-lake) cell in the top layer, using the vertical hydraulic conductivity for the saturated vertical conductivity (`vks`), with rainfall (`finf`) and land evapotranspiration (**ET**, water returned to the atmosphere from soil and plants; `pet` is its potential rate) applied each stress period.

In [ ]:
# vertical hydraulic conductivity defines vks for the uzf package
k33 = gwf.npf.k33.array

In [ ]:
# UZF package data (one entry per non-lake cell in layer 1)
# <ifno> <cellid> <landflag> <ivertcon> <surfdep> <vks> <thtr> <thts> <thti> <eps>
packagedata = []
ifno = 0
for i in range(nrow):
    for j in range(ncol):
        if lake_location[i, j] == 1:
            continue
        cellid = (0, i, j)
        packagedata.append((ifno, cellid, 1, 0, 1.0, k33[cellid], 0.05, 0.25, 0.1, 3.5))
        ifno += 1

In [ ]:
# UZF stress period data with rainfall (finf) and land evapotranspiration (pet)
# <ifno> <finf> <pet> <extdp> <extwc> <ha> <hroot> <rootact>
uzf_spd = {}
rain_tag = "PRCP (Inches)"
pet_tag = "land et (inches)"
for n in range(nper):
    row = temporal_df.iloc[n]
    rain = float(row[rain_tag]) * IN2FT
    pet = float(row[pet_tag]) * IN2FT
    uzf_spd[n] = [
        (ifno, rain, pet, 10.0, -999.0, -999.0, -999.0, -999.0)
        for ifno, *_ in packagedata
    ]

In [ ]:
# replace the recharge package with UZF (idempotent: drop any prior UZF/RCH first)
drop_packages(gwf, "UZF-1", "RCH_0")
uzf = flopy.mf6.ModflowGwfuzf(
    gwf,
    pname="UZF-1",
    mover=True,
    simulate_et=True,
    nuzfcells=len(packagedata),
    packagedata=packagedata,
    perioddata=uzf_spd,
)
gwf.get_package_list()

## Write and run

Write the updated advanced model back to `models/` and run it to confirm the UZF package is valid.

In [ ]:
sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"
print("UZF advanced model ran successfully")

## Recap

In this notebook you replaced the Recharge (**RCH**) package with the Unsaturated Zone Flow (**UZF**) package, which routes infiltration through an unsaturated column before it reaches the water table, then wrote the updated advanced model back to `models/` and ran it. Build the other advanced packages (MAW, SFR, LAK, MVR) with their own notebooks, then run `advanced-packages-processing` to see the combined results.